# Build Your First Voice Agent using Pipecat

This cookbook walks through building a **real-time, multilingual voice agent** with **[Pipecat](https://docs.pipecat.ai)** and **[Sarvam AI](https://docs.sarvam.ai)**.

By the end of this notebook you will have a voice agent that can:

- **Listen** to a caller speaking any of 10 Indian languages (or English)
- **Understand** the request with a Sarvam chat model
- **Respond** back in a natural-sounding voice

Pipecat is a Python framework for building real-time, multimodal pipelines out of composable processors (transport in, STT, LLM, TTS, transport out). Sarvam plugs into that pipeline as the speech-to-text (**Saarika/Saaras**), LLM (**sarvam-30b** / **sarvam-105b**), and text-to-speech (**Bulbul**) processors.

> **Note on running this notebook:** A voice agent is a long-running process that waits for a caller to join a room, so it can't be "run to completion" in a single notebook cell. This notebook builds the agent file for you (`agent.py`) using `%%writefile`, then shows the exact terminal commands to run and test it.

## Overview

| | |
|---|---|
| **Transport** | [Daily](https://daily.co) (WebRTC rooms) or browser WebRTC |
| **Speech-to-text** | Sarvam **Saarika/Saaras** (`SarvamSTTService`) |
| **LLM** | Sarvam chat models (`SarvamLLMService`, default `sarvam-105b`) |
| **Text-to-speech** | Sarvam **Bulbul v3** (`SarvamTTSService`) |
| **Languages** | 10 Indian languages + English, with auto-detect |
| **Lines of code** | ~80 |

### Quick overview of the steps

1. Get a Sarvam API key.
2. Install `pipecat-ai` with the `daily` and `sarvam` extras.
3. Create a `.env` file with your API key.
4. Write the agent (`agent.py`).
5. Run it — Pipecat spins up a Daily room and gives you a URL to join.

## Prerequisites

- Python 3.9 or higher.
- A [Sarvam AI](https://dashboard.sarvam.ai) API key.

## 1. Install dependencies

The `daily` extra adds the [Daily](https://daily.co) WebRTC transport (Pipecat also supports plain WebRTC without an extra account); the `sarvam` extra adds the Sarvam STT/LLM/TTS services. `loguru` gives readable structured logs.

In [ ]:
%pip install -Uqq "pipecat-ai[daily,sarvam]" python-dotenv loguru

## 2. Configure your Sarvam API key

Get your key from [dashboard.sarvam.ai](https://dashboard.sarvam.ai). Never commit real API keys to version control — the cell below writes a `.env.example` template; copy it to `.env` and fill in your real key locally:

```bash
cp .env.example .env
```

In [ ]:
%%writefile .env.example
# Copy this file to `.env` and fill in your real credentials.
# Do not commit `.env` to version control.

SARVAM_API_KEY=your_sarvam_api_key

## 3. Write the voice agent

The pipeline below wires together, in order: transport input → STT → conversation context (user turn) → LLM → TTS → transport output → conversation context (assistant turn). This is Pipecat's standard voice-agent shape.

`%%writefile` saves this cell's contents to `agent.py` in the current directory.

In [ ]:
%%writefile agent.py
import os

from dotenv import load_dotenv
from loguru import logger
from pipecat.frames.frames import LLMRunFrame
from pipecat.pipeline.pipeline import Pipeline
from pipecat.pipeline.runner import PipelineRunner
from pipecat.pipeline.task import PipelineTask
from pipecat.processors.aggregators.llm_context import LLMContext
from pipecat.processors.aggregators.llm_response_universal import (
    LLMContextAggregatorPair,
)
from pipecat.runner.types import RunnerArguments
from pipecat.runner.utils import create_transport
from pipecat.services.sarvam.llm import SarvamLLMService
from pipecat.services.sarvam.stt import SarvamSTTService
from pipecat.services.sarvam.tts import SarvamTTSService
from pipecat.transports.base_transport import TransportParams
from pipecat.transports.daily.transport import DailyParams

load_dotenv(override=True)


async def bot(runner_args: RunnerArguments) -> None:
    """Main bot entry point."""

    # Create the transport (supports both Daily and browser WebRTC).
    transport = await create_transport(
        runner_args,
        {
            "daily": lambda: DailyParams(audio_in_enabled=True, audio_out_enabled=True),
            "webrtc": lambda: TransportParams(
                audio_in_enabled=True, audio_out_enabled=True
            ),
        },
    )

    # Initialize the Sarvam AI services.
    stt = SarvamSTTService(api_key=os.getenv("SARVAM_API_KEY"))
    tts = SarvamTTSService(api_key=os.getenv("SARVAM_API_KEY"))
    llm = SarvamLLMService(
        api_key=os.getenv("SARVAM_API_KEY"),
        settings=SarvamLLMService.Settings(model="sarvam-105b"),
    )

    # Set up conversation context.
    messages = [
        {
            "role": "system",
            "content": "You are a friendly AI assistant. Keep your responses brief and conversational.",
        },
    ]
    context = LLMContext(messages)
    context_aggregator = LLMContextAggregatorPair(context)

    # Build the pipeline: audio in -> STT -> LLM -> TTS -> audio out.
    pipeline = Pipeline(
        [
            transport.input(),
            stt,
            context_aggregator.user(),
            llm,
            tts,
            transport.output(),
            context_aggregator.assistant(),
        ]
    )

    task = PipelineTask(pipeline)

    @transport.event_handler("on_client_connected")
    async def on_client_connected(transport, client) -> None:
        logger.info("Client connected")
        messages.append(
            {"role": "system", "content": "Say hello and briefly introduce yourself."}
        )
        await task.queue_frames([LLMRunFrame()])

    @transport.event_handler("on_client_disconnected")
    async def on_client_disconnected(transport, client) -> None:
        logger.info("Client disconnected")
        await task.cancel()

    runner = PipelineRunner(handle_sigint=runner_args.handle_sigint)
    await runner.run(task)


if __name__ == "__main__":
    from pipecat.runner.run import main

    main()

## 4. Run and test your agent

For Daily transport, from a terminal in the same directory as `agent.py` and `.env`:

```bash
python agent.py
```

The agent creates a Daily room and prints a URL. Open that URL in your browser and start speaking — your voice agent will listen and respond!

## Customization examples

Swap these blocks into the service-initialization section of `agent.py` and re-run.

### Hindi voice agent

```python
stt = SarvamSTTService(
    api_key=os.getenv("SARVAM_API_KEY"),
    language="hi-IN",  # Hindi
    model="saaras:v3",
    mode="transcribe",
)

tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    target_language_code="hi-IN",
    model="bulbul:v3",
    speaker="simran",  # or: priya, ishita, kavya, aditya, anand, rohan
)

llm = SarvamLLMService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamLLMService.Settings(model="sarvam-105b"),
)
```

### Tamil voice agent

```python
stt = SarvamSTTService(
    api_key=os.getenv("SARVAM_API_KEY"),
    language="ta-IN",
    model="saaras:v3",
    mode="transcribe",
)

tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    target_language_code="ta-IN",
    model="bulbul:v3",
    speaker="shubh",
)
```

### Multilingual agent (auto-detect)

```python
stt = SarvamSTTService(
    api_key=os.getenv("SARVAM_API_KEY"),
    language="unknown",  # auto-detects language
    model="saaras:v3",
    mode="transcribe",
)

tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    target_language_code="en-IN",
    model="bulbul:v3",
    speaker="anand",
)
```

### Speech-to-English agent (Saaras)

Saarika transcribes speech to text in the same language; Saaras translates speech directly to English text. Use Saaras when the caller speaks an Indian language but you want to process and respond in English — it auto-detects the source language and translates the spoken content directly.

```python
# Caller speaks Hindi -> Saaras converts to English -> LLM processes -> replies in English
stt = SarvamSTTService(
    api_key=os.getenv("SARVAM_API_KEY"),
    model="saaras:v3",
    mode="translate",  # speech-to-English translation
)

tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    target_language_code="en-IN",
    model="bulbul:v3",
    speaker="aditya",
)

llm = SarvamLLMService(
    api_key=os.getenv("SARVAM_API_KEY"),
    settings=SarvamLLMService.Settings(model="sarvam-105b"),
)
```

## Available options

### Language codes

| Language | Code |
|---|---|
| English (India) | `en-IN` |
| Hindi | `hi-IN` |
| Bengali | `bn-IN` |
| Tamil | `ta-IN` |
| Telugu | `te-IN` |
| Gujarati | `gu-IN` |
| Kannada | `kn-IN` |
| Malayalam | `ml-IN` |
| Marathi | `mr-IN` |
| Punjabi | `pa-IN` |
| Odia | `od-IN` |
| Auto-detect | `unknown` |

### Speaker voices (Bulbul v3)

- **Male (23):** shubh (default), aditya, rahul, rohan, amit, dev, ratan, varun, manan, sumit, kabir, aayan, ashutosh, advait, anand, tarun, sunny, mani, gokul, vijay, mohit, rehan, soham
- **Female (14):** ritu, priya, neha, pooja, simran, kavya, ishita, shreya, roopa, tanya, shruti, suhani, kavitha, rupali

### TTS additional parameters

```python
tts = SarvamTTSService(
    api_key=os.getenv("SARVAM_API_KEY"),
    target_language_code="en-IN",
    model="bulbul:v3",
    speaker="shubh",
    pace=1.0,                 # range: 0.5 to 2.0
    speech_sample_rate=24000,  # 8000, 16000, 22050, 24000 Hz (default); v3 REST API also supports 32000, 44100, 48000 Hz
)
```

## Understanding the pipeline

Pipecat uses a **pipeline architecture** where data flows through a series of processors:

```
User Audio -> STT -> Context Aggregator -> LLM -> TTS -> Audio Output
```

1. **Transport input** — receives audio from the user.
2. **STT (speech-to-text)** — converts audio to text using Sarvam's Saarika/Saaras.
3. **Context aggregator (user)** — adds the user's message to the conversation context.
4. **LLM** — generates a response using Sarvam.
5. **TTS (text-to-speech)** — converts the response to audio using Sarvam's Bulbul.
6. **Transport output** — sends the audio back to the user.
7. **Context aggregator (assistant)** — saves the assistant's response to context.

In [ ]:
# Quick reference: the pipeline stage order used above, and the languages/voices it supports.
PIPELINE_STAGES = [
    "transport.input()",
    "SarvamSTTService (speech -> text)",
    "context_aggregator.user()",
    "SarvamLLMService (generate reply)",
    "SarvamTTSService (text -> speech)",
    "transport.output()",
    "context_aggregator.assistant()",
]

for i, stage in enumerate(PIPELINE_STAGES, start=1):
    print(f"{i}. {stage}")

## Pro tips

- Use `language="unknown"` to automatically detect the language — great for multilingual scenarios.
- Sarvam's models understand code-mixing, so your agent can naturally handle Hinglish, Tanglish, and other mixed languages.
- Adjust `pace` to customize the voice delivery speed.
- Use `sarvam-30b` for faster responses, or `sarvam-105b` for more complex conversations.

## Troubleshooting

- **API key errors** — Check that all keys are in your `.env` file and that the file is in the same directory as `agent.py`.
- **Module not found** — Re-run the install cell for your operating system.
- **Poor transcription** — Try `language="unknown"` for auto-detection, or specify the correct language code (`en-IN`, `hi-IN`, etc.).
- **Connection issues** — Ensure you have a stable internet connection and the transport is properly configured.

## Additional resources

- [Sarvam AI Documentation](https://docs.sarvam.ai)
- [Pipecat Documentation](https://docs.pipecat.ai)
- [Pipecat Sarvam LLM Service](https://docs.pipecat.ai/api-reference/server/services/llm/sarvam)
- [Pipecat GitHub Repository](https://github.com/pipecat-ai/pipecat)
- [Daily.co Documentation](https://docs.daily.co)

## Need help?

- Sarvam Support: developer@sarvam.ai
- Community: [Join the Discord Community](https://discord.com/invite/5rAsykttcs)